# Stock Market Data Ingestion
Pulls daily OHLCV stock data for 37 tickers from Yahoo Finance API and lands raw Parquet files in the bronze layer of Azure Data Lake Storage Gen2.

**Schedule:** Run daily after market close (weekdays only)
**Output:** Parquet files at /Volumes/stock_pipeline/bronze/stock_data/{ticker}/date={today}/

In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

## Imports and Configuration
Import required libraries and define the list of 37 tickers organized by sector.

In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime

TICKERS = [
    # FAANG
    "META", "AAPL", "AMZN", "NFLX", "GOOGL",
    # Big Tech
    "MSFT", "NVDA", "AMD", "INTC", "TSLA",
    # Finance
    "JPM", "GS", "BAC", "MS", "V", "MA", "AXP", "BLK",
    # Healthcare
    "JNJ", "PFE", "UNH", "ABBV", "MRK",
    # Retail
    "WMT", "TGT", "NKE", "MCD", "SBUX", "COST",
    # Energy
    "XOM", "CVX", "COP",
    # ETFs
    "SPY", "QQQ", "DIA", "IWM", "VTI"
]

print(f"Tracking {len(TICKERS)} tickers")

## Fetch Stock Data
Pulls the most recent 1 year of daily OHLCV data for a single ticker from Yahoo Finance.
Using period="1y" ensures we always have a full year of history even when running for the first time.

In [0]:
def fetch_stock_data(ticker):
    """Fetch last 1 year of daily OHLCV data for a ticker"""
    stock = yf.Ticker(ticker)
    df = stock.history(period="1y", interval="1d", auto_adjust=True)
    
    if df.empty:
        raise ValueError(f"No data returned for {ticker}")
    
    df.reset_index(inplace=True)
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.normalize()
    df["Ticker"] = ticker
    df = df[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]]
    
    return df

print("Fetch function defined")

## Upload to Bronze Layer
Writes each ticker's DataFrame as a Parquet file to the bronze layer in ADLS Gen2.
Files are partitioned by ticker and date for efficient querying.
Path format: /Volumes/stock_pipeline/bronze/stock_data/{ticker}/date={today}/

In [0]:
def upload_to_bronze(df, ticker):
    """Write DataFrame as Parquet to bronze volume"""
    today = datetime.today().strftime("%Y-%m-%d")
    path = f"/Volumes/stock_pipeline/bronze/stock_data/{ticker}/date={today}/"
    
    # Convert pandas DataFrame to Spark DataFrame
    spark_df = spark.createDataFrame(df)
    
    # Write as parquet
    spark_df.write \
        .mode("overwrite") \
        .parquet(path)
    
    print(f"  [OK] {ticker} -> {path} ({len(df)} rows)")

print("Upload function defined")

## Run Ingestion
Loops through all 37 tickers, fetches data from Yahoo Finance, and uploads to the bronze layer.
Failed tickers are tracked and reported at the end without stopping the pipeline.

In [0]:
print(f"Starting ingestion for {len(TICKERS)} tickers at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 60)

succeeded = []
failed = []

for ticker in TICKERS:
    try:
        df = fetch_stock_data(ticker)
        upload_to_bronze(df, ticker)
        succeeded.append(ticker)
    except Exception as e:
        print(f"  [FAIL] {ticker}: {e}")
        failed.append((ticker, str(e)))

print("-" * 60)
print(f"Ingestion complete: {len(succeeded)}/{len(TICKERS)} succeeded")
if failed:
    print("Failed tickers:")
    for ticker, err in failed:
        print(f"  {ticker}: {err}")

## Verify Ingestion
Confirm today's files were written successfully to the bronze layer.

In [0]:
today = datetime.today().strftime("%Y-%m-%d")
print(f"Verifying files for {today}:")
print("-" * 60)

files = dbutils.fs.ls("/Volumes/stock_pipeline/bronze/stock_data/")
for f in files:
    ticker = f.name.replace("/", "")
    try:
        ticker_files = dbutils.fs.ls(f"/Volumes/stock_pipeline/bronze/stock_data/{ticker}/date={today}/")
        print(f"  [OK] {ticker} — {len(ticker_files)} file(s)")
    except:
        print(f"  [MISSING] {ticker} — no file for today")